# FleetLogix · Avance #4 — Cloud Architecture (AWS)

Propuesta académica de arquitectura (sin desplegar AWS). El objetivo es explicar un flujo de datos móviles de entregas en tiempo real, persistencia, seguridad, observabilidad y costo teórico inferior a US$50/mes.

**Ver también:** `Documentacion/Avance_4/ARQUITECTURA_TECNICA.md` y `aws_architecture_diagram.png`.


## Arquitectura propuesta

App móvil → API Gateway HTTP API → Lambda → DynamoDB (estado actual) y S3 (eventos históricos). EventBridge → Lambda batch → RDS PostgreSQL. CloudWatch recibe métricas y logs. La migración inicial de PostgreSQL local a RDS es planificada, no ejecutada.

![Diagrama](../Documentacion/Avance_4/aws_architecture_diagram.png)


## Funciones de ejemplo

Cuatro funciones puras de Python: validar estado completado, calcular ETA simple, detectar distancia a waypoint y normalizar eventos. No necesitan cuenta de AWS.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'Scripts' / 'Avance_4'))
from lambda_functions.delivery_logic import (
    check_delivery_completed, estimate_eta, detect_route_deviation, validate_event
)
event = {'event_id':'demo-1','delivery_id':'101',
         'event_type':'delivery_status','status':'delivered',
         'occurred_at':'2026-11-17T14:00:00Z'}
print(check_delivery_completed(event))
print(estimate_eta({'remaining_distance_km':10,'average_speed_kmh':40}))


## Decisiones y restricciones

- RDS privado Single-AZ con backups y cifrado; DynamoDB para última actualización de entrega; S3 particionado por fecha para historial.
- IAM mínimo; API autenticada; CloudWatch y alarmas; no usar cuenta root.
- Control futuro de duplicados y fallos parciales entre DynamoDB y S3.
- Estimación de laboratorio: **US$24,83/mes**, sin NAT ni Multi-AZ, escenario de bajo tráfico; confirmar tarifas regionales en AWS Pricing Calculator.
- Avance orientado a **diseño/documentación**, no a despliegue de infraestructura real.
